# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id` fields as per the Croissant specification.

### Dataset Source
The dataset source is a Croissant schema located at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

# Print collection time and location as examples
print(f"Data Collection Timeframe: {getattr(metadata, 'dataCollectionTimeframe', None)}")
print(f"Spatial Coverage: {getattr(metadata, 'spatialCoverage', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Available record sets: {getattr(metadata, 'recordSet', None)}")

## 2. Data Overview
Review available record sets and their structure, referencing each by `@id`.

> Note: If record set structure is not present in metadata, we will enumerate programmatically using `dataset.record_sets`.

In [ ]:
# List all record sets in this Croissant dataset
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record set @id's found: {record_set_ids}\n")
for rs in dataset.record_sets:
    print(f"--- Record Set @id: {rs.get('@id')} ---")
    print(f"Name: {rs.get('name', rs.get('@id'))}")
    # Show fields by @id, if present
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("Fields @ids:", [f.get('@id', 'unknown') if isinstance(f, dict) else str(f) for f in fields])
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing record set and field `@id` values discovered above.

In [ ]:
# Extract data from all available record sets into dataframes
dataframes = {}
for rec_set in record_set_ids:
    records = list(dataset.records(record_set=rec_set))
    df = pd.DataFrame(records)
    dataframes[rec_set] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {rec_set}")
    print(f"Columns (@id): {list(df.columns)}\n")

# For demonstration, select the first record set
if record_set_ids:
    example_record_set = record_set_ids[0]
    print(f"First 5 rows of RecordSet @id: {example_record_set}")
    display(dataframes[example_record_set].head())
else:
    print("No record sets available to display.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate typical data processing such as filtering records, normalizing a numeric field, and grouping by categorical fields—all referenced by their `@id`.

> If you are unsure about the numeric/categorical field `@id`, print a sample or describe the dataframe to decide. Replace the `<numeric_field_id>` and `<group_field_id>` accordingly below.

In [ ]:
# Choose which record set and columns to perform EDA on
df = dataframes[example_record_set]

# Show datatypes and statistics to help select fields
print("DataFrame dtypes:")
print(df.dtypes)
print("\nSummary statistics:")
print(df.describe(include='all'))

# For this dataset (logistic regression outputs), likely numeric columns may include: log_likelihood, coef, stderr, pval etc.
# Detect numeric columns programmatically or set manually:
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("\nNumeric columns detected:", numeric_candidates)

# Pick the first detected numeric field @id, if any
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    # Choose a group field; pick first object/categorical column
    potential_group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    group_field_id = potential_group_fields[0] if potential_group_fields else None
    print(f"Using numeric_field_id: {numeric_field_id}")
    print(f"Using group_field_id: {group_field_id}")
    
    # Set a threshold for filtering
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (total {len(filtered_df)} records):")
    print(filtered_df.head())
    
    # Create normalized field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping by group_field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_" + numeric_field_id)
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the selected group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If grouped, show boxplot by group_field_id
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to use `mlcroissant` to load, examine, and process the FAIR<sup>2</sup> dataset package containing ordered logistic regression outputs for knowledge adoption prediction in Northern Kenya. All dataset components—record sets, fields, and columns—were referenced using their Croissant `@id` identifiers to ensure reproducibility and structural clarity.

- We examined dataset metadata, such as the data collection context and timeframe.
- All available record sets and their fields were enumerated by `@id`.
- Data was loaded into DataFrames for inspection and processed with common EDA techniques such as filtering, normalization, and group-wise summarization.
- Distributions and relationships in the data were visualized for selected (numeric, categorical) fields.

This workflow can be adapted to any Croissant-compatible dataset using the same programmatic approach. Always refer to fields and entities by their `@id` as best practice when working with the Croissant standard.